In [1]:
import pandas as pd

# Loading in data and renaming columns
def load_data(file_path):
    df = pd.read_csv(file_path, header=None, sep="\t")

    df = df.rename(columns={
        0: "id",
        1: "modified",
        2: "sequence",
        3: "position"
    })
    df["position"] = df["position"].astype("Int64")
    return df



file = "/home/callum/projects/sno_analysis/data/raw/SNO_combined.txt"
df = load_data(file)

print("duplicates...")
print(df.duplicated().sum())


print(df.dtypes)

# function exploring dataset and giving basic info and counts
# def explore_data(df):
#     """Print a summary of shape, dtypes, nulls, and key value counts."""
#     print("Shape:", df.shape)
#     print("\nDtypes:\n", df.dtypes)
#     print("\nMissing values per column:\n", df.isnull().sum())
#     print("\nUnique proteins:", df["id"].nunique())
#     print(f"\nClass balance (modified): {df['modified'].value_counts()}\n")
# explore_data(df)


# --- Identifying conflicting sites ---
# for each id and position (each unique site), how many unique modified values are there?
# ie are there both positives and negatives for the same site
site_counts = df.groupby(["id", "position"])["modified"].nunique()
# this gives a series with id and position as index and 1 column - the number of unique entries

# filter for values greater than 1 - ie 2 different modification values present for the same site
conflicting_sites = site_counts[site_counts > 1]
print(conflicting_sites)

# reset the index to make id and position columns in a proper dataframe
conflicting_sites_df = conflicting_sites.reset_index()

# rename the count column from this df to identify column that marks conflicts
conflicting_sites_df = conflicting_sites_df.rename(columns={"modified": "conflicts"})
print(conflicting_sites_df)


# --- Merging conflict info back onto the full dataset ---
print("merging dataframes...")
merged = pd.merge(df, conflicting_sites_df, on=["id", "position"], how="left", indicator=True)

# confirming number of matched rows
# there are 334 - conflicting pairs have an entry for both the 0 and the 1 modification
print(merged["_merge"].value_counts())


# --- Filtering: keep non-conflicting rows, plus only the positive row for conflicts ---
before = int(merged.shape[0])

condition1 = merged["_merge"] == "left_only"
condition2 = (merged["_merge"] == "both") & (merged["modified"] == 1)
merged_filtered = merged[condition1 | condition2]

after = int(merged_filtered.shape[0])
print(f"Rows dropped: {before - after}")

# drop the helper columns now that filtering is done
to_drop = ["_merge", "conflicts"]
merged_filtered_clean = merged_filtered.drop(to_drop, axis="columns")

print(merged_filtered_clean.head())
print(merged_filtered_clean.shape[0])



duplicates...
0
id            str
modified    int64
sequence      str
position    Int64
dtype: object
id      position
A2ASS6  13296       2
        14760       2
        18679       2
        21561       2
        21780       2
                   ..
Q9WUM4  456         2
Q9WV60  14          2
        317         2
Q9Z0E0  415         2
        725         2
Name: modified, Length: 167, dtype: int64
         id  position  conflicts
0    A2ASS6     13296          2
1    A2ASS6     14760          2
2    A2ASS6     18679          2
3    A2ASS6     21561          2
4    A2ASS6     21780          2
..      ...       ...        ...
162  Q9WUM4       456          2
163  Q9WV60        14          2
164  Q9WV60       317          2
165  Q9Z0E0       415          2
166  Q9Z0E0       725          2

[167 rows x 3 columns]
merging dataframes...
_merge
left_only     23897
both            334
right_only        0
Name: count, dtype: int64
Rows dropped: 167
       id  modified                         

# made into a function


In [2]:
def resolve_label_conflicts(df, group_cols, label_col, keep_value=1):
    """
    For rows sharing the same group_cols (e.g. id + position), where label_col
    has more than one distinct value, keep only the row where label_col == keep_value.
    Rows with no conflict are left untouched.
    """
    # count distinct label values per group
    label_counts = df.groupby(group_cols)[label_col].nunique()
    conflicting_groups = label_counts[label_counts > 1].reset_index()
    conflicting_groups = conflicting_groups.rename(columns={label_col: "n_unique_labels"})

    # flag which rows belong to a conflicting group
    merged = pd.merge(df, conflicting_groups, on=group_cols, how="left", indicator=True)

    # keep: non-conflicting rows, plus only the "keep_value" row within conflicting groups
    no_conflict = merged["_merge"] == "left_only"
    resolved_conflict = (merged["_merge"] == "both") & (merged[label_col] == keep_value)

    result = merged[no_conflict | resolved_conflict].drop(columns=["_merge", "n_unique_labels"])
    return result.reset_index(drop=True)

In [3]:
# filtering for sites

print(merged_filtered_clean.head())
positive_condition = merged_filtered_clean["modified"] == 1
negative_condition = merged_filtered_clean["modified"] == 0 

positive_sites = merged_filtered_clean[positive_condition]
negative_sites = merged_filtered_clean[negative_condition]

df = merged_filtered_clean
print(merged_filtered_clean.shape)
both = merged_filtered_clean.shape[0]
positive = positive_sites.shape[0]
negative = negative_sites.shape[0]

print(positive_sites.shape)
print(negative_sites.shape)

print(f"{positive + negative == both}")


positive_sites2 = merged_filtered_clean.query("modified == 1")
negative_sites2 = merged_filtered_clean.query("modified == 0")

       id  modified                                   sequence  position
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735
4  P0C8J8         0  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS        17
(24064, 4)
(3774, 4)
(20290, 4)
True


In [4]:
# counting / filtering for proteins based on how many sites they have - can identify highly modified proteins
series = df.groupby(["id"]).size()
print(series)

#make a dataframe
#series = series.reset_index()
print(series)

print(series.shape)

#filter based on 5 modifications 
mask = series >= 5
excluded_mask = series < 5
# apply the mask
series_filtered = series[mask]
series_filtered = series_filtered.reset_index()

excluded = series[excluded_mask]
excluded = excluded.reset_index()

# # check how many proteins appear 5 or more times

print(f"proteins > 5:{series_filtered.shape}")

# merge using array of protein ids 

df_merged = pd.merge(df, series_filtered, on = "id", how = "inner", indicator = False)

excluded_rows = pd.merge(df, excluded, on="id", how="inner")

print("================##===============")
print(f"proteins > 5:{series_filtered.shape}")
print(f"original df:{df.shape}")
print(f"new dataframe: {df_merged.shape}")
print(f"ecluded: {excluded.shape}")

print("================##===============")

# not sure how to sanity check

print(f"{df_merged.shape[0] + excluded_rows.shape[0] == df.shape[0]}")


id
A2A432     12
A2AAJ9    237
A2AJB2      3
A2AJI0      6
A2AN08    122
         ... 
Q9Z2W0     10
Q9Z2X1      6
Q9Z2Z6      6
Q9ZNZ7     25
Q9ZVA2     12
Length: 2217, dtype: int64
id
A2A432     12
A2AAJ9    237
A2AJB2      3
A2AJI0      6
A2AN08    122
         ... 
Q9Z2W0     10
Q9Z2X1      6
Q9Z2Z6      6
Q9ZNZ7     25
Q9ZVA2     12
Length: 2217, dtype: int64
(2217,)
proteins > 5:(1572, 2)
================##===============
proteins > 5:(1572, 2)
original df:(24064, 4)
new dataframe: (22316, 5)
ecluded: (645, 2)
================##===============
True


In [ ]:
if 0 in df_merged.columns:
    df_merged = df_merged.drop(columns=[0])

print(df_merged.head())
df = df_merged



def get_top_n_proteins(df, n, colname):
    series = df.groupby(colname).size()
    
    # turn into df so we can use it 
    series = series.reset_index()
    series = series.rename(columns={0: "occurrences"})

    # now applying .nlargest - this sorts AND selects in one
    top = series.nlargest(n = n, columns = "occurrences")

    # merging dataframe to create master table of top 20 proteins
    top_merged = pd.merge(df, top, on = "id", how = "inner", indicator= False)

    # sanity check - should be 20 proteins
    print(f"number of unique proteins in final df: {top_merged["id"].nunique()}")
    
    return top_merged


       id  modified                                   sequence  position
0  Q9Y2Z0         1  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL        88
1  Q14584         0  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK       441
2  P00390         0  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG       328
3  Q5UIP0         0  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN      1735
5  Q9NYY3         1  TITYQPAASTKMCEQALGKGCGADSKKKRPPQPPEESQPPQ        26


In [29]:
print(df.shape)

tm = get_top_n_proteins(df, 20, "id")
print(tm)

(22315, 4)
number of unique proteins in final df: 20
          id  modified                                   sequence  position  \
0     P00533         0  QKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSC       506   
1     P21817         0  ERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPK       936   
2     P98156         0  IQCGSGECIHKKWRCDGDPDCKDGSDEVNCPSRTCRPDQFE       264   
3     A2ASS6         0  SEPSDPAVAGDPIYPPGPPSCPEVKDKTKSSISLAWKPPAK     17499   
4     O00468         0  RSCRVNPRTRRPEMLLRPESCPARQAPVCGDDGVTYENDCV       349   
...      ...       ...                                        ...       ...   
3268  Q7TMY8         0  TKETNCESDRERGSKQACSPCSSQSSSSGICTDFWDLLVKL      3376   
3269  Q7TMY8         0  IGEILIQGLTEDMVTVLIRACVSMLGVPVDPDTLHATLRLC      1762   
3270  Q7TMY8         0  TRDTVLKLLLNGARHLGYTLCKQIGTLLAELREYNLEQQRR      3636   
3271  P19137         0  KNCDRCKPGFYNLKERNPEGCSECFCFGVSGVCDSLTWSIS       507   
3272  P19137         0  DTGLGCVPCNCSVEGSVSDNCTEEGQCHCGPGVSGKQCDRC       970   